### Utility Functions

In [2]:
# This part helps with stability on multi-GPU systems with great inbalance between the GPUs (eg. integrated vs discrete gpu)
# IMPORTANT must be done before importing torch, else session must be restarted
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import torch

for i in range(torch.cuda.device_count()):
    free = torch.cuda.mem_get_info(i)[0]
    total = torch.cuda.mem_get_info(i)[1]

    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Free:  {free / 1024**3:.2f} GB")
    print(f"  Total: {total / 1024**3:.2f} GB")

GPU 0: NVIDIA GeForce RTX 3070 Laptop GPU
  Free:  7.50 GB
  Total: 7.66 GB


In [4]:
from util import clear_folder

# Clears results of the last run, not necessary, just to reduce folder size

clear_folder("./results")

In [5]:
from util import clear_cuda_cache

# If model gets stuck during training, uncomment the following line

clear_cuda_cache()

# Training Pipeline

## Load Dataset

This portion loads dataset, and assigns id for each label

#### English

This loads english version of the dataset

In [6]:
from util import load_datasets_from_hf

label2id={"ham": 0, "spam": 1}

train_dataset, val_dataset, test_dataset, id2label = load_datasets_from_hf(
    dataset_name="daviiiidcpp1/sms-spam-combined",
    split="train",
    label2id=label2id
)


Loaded datasets (train=44323, val=5540, test=5541)


#### Slovenian

This loads slovenian version of the dataset

In [12]:
from util import load_datasets_from_hf

label2id={"ham": 0, "spam": 1}

train_dataset, val_dataset, test_dataset, id2label = load_datasets_from_hf(
    dataset_name="daviiiidcpp1/sms-spam-slovene",
    split="train",
    label2id=label2id
)


Generating train split: 100%|██████████| 55404/55404 [00:00<00:00, 810159.25 examples/s]


Loaded datasets (train=44323, val=5540, test=5541)


Map: 100%|██████████| 5541/5541 [00:00<00:00, 13879.90 examples/s]


## Model settings 

### XLM-RoBERTA

In [ ]:
from util import tokenize_datasets, load_model_and_tokenizer, build_trainer


model_name = "FacebookAI/xlm-roberta-base"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


### TinyBert

In [ ]:
from util import tokenize_datasets, load_model_and_tokenizer, build_trainer


model_name = "huawei-noah/TinyBERT_General_4L_312D"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


### mBERT

In [5]:
from util import tokenize_datasets, load_model_and_tokenizer, build_trainer


model_name = "bert-base-multilingual-cased"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8608.71it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

## Train

In [17]:
from datetime import datetime
from util import clear_folder

clear_folder("./results")

trainer.train()

model_save_folder = "trained_models" 

model_save_name = model_name.split("/")[-1] + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
trainer.save_model(os.path.join(model_save_folder, model_save_name))
tokenizer.save_pretrained(os.path.join(model_save_folder, model_save_name))

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.123732,0.200549,0.963899,0.955209,0.947826,0.951503
2,0.112166,0.171142,0.969856,0.958554,0.960870,0.959710
3,0.039785,0.210840,0.970758,0.959981,0.961836,0.960907


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

('trained_models\\bert-base-multilingual-cased2026-05-07_22-06-34\\tokenizer_config.json',
 'trained_models\\bert-base-multilingual-cased2026-05-07_22-06-34\\tokenizer.json')

In [7]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
No log,0.678501,0,0.628881,0.602941,0.019807,0.038354


{'eval_loss': 0.6785014271736145,
 'eval_accuracy': 0.6288808664259928,
 'eval_precision': 0.6029411764705882,
 'eval_recall': 0.019806763285024155,
 'eval_f1': 0.03835360149672591}

### Data Visualization

In [ ]:
import graphs
import importlib

importlib.reload(graphs)

graphs.generate_all_plots("./results", "./graphs" + "/" + model_name.split("/")[-1])

# Testing pipeline

In [8]:
from util import load_latest_trained_model_and_tokenizer, build_trainer

model, tokenizer, latest_dir = load_latest_trained_model_and_tokenizer(
    trained_models_root="./trained_models",
    num_labels=2,
 )
print("Loaded:", latest_dir)


trainer = build_trainer(model, train_dataset, val_dataset)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3703.00it/s]


Loaded: ./trained_models\bert-base-multilingual-cased2026-05-07_22-06-34


## Load a previously trained model (skip training)

If you already trained a model once, you can reload it from `./trained_models` and run evaluation/zero-shot testing without training again.

# Zero-shot testing (EN -> SL)

Evaluate the model trained on the English dataset on the Slovenian dataset without any further training.

In [6]:
from util import load_datasets_from_hf, tokenize_dataset


sl_train_ds, sl_val_ds, sl_test_ds, sl_id2label = load_datasets_from_hf(
    dataset_name="daviiiidcpp1/sms-spam-slovene",
    split="train",
    label2id=label2id,
    train_size=0.8,
    val_size=0.1,
    seed=42,
 )

sl_test_ds_tok = tokenize_dataset(sl_test_ds, tokenizer)

trainer.eval_dataset = sl_test_ds_tok

zero_shot_metrics = trainer.evaluate()


Loaded datasets (train=44323, val=5540, test=5541)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
No log,0.738037,0,0.367262,0.367016,0.967492,0.532159


{'eval_loss': 0.738036572933197,
 'eval_accuracy': 0.36726222703483125,
 'eval_precision': 0.3670163813730904,
 'eval_recall': 0.9674915089762252,
 'eval_f1': 0.5321590605817987}

In [9]:
from util import predict_labels
import graphs

labels_in_order = [sl_id2label[i] for i in range(len(sl_id2label))]

preds, labels = predict_labels(trainer, sl_test_ds_tok)
graph_dir = os.path.join("./graphs", os.path.basename(latest_dir))
graphs.plot_confusion_matrix(
    y_true=labels,
    y_pred=preds,
    labels=labels_in_order,
    save_path=os.path.join(graph_dir, "zero_shot_en_to_sl_confusion_matrix.png"),
)
print(f"Saved confusion matrix to {os.path.join(graph_dir, 'zero_shot_en_to_sl_confusion_matrix.png')}")

Saved confusion matrix to ./graphs\bert-base-multilingual-cased2026-05-07_22-06-34\zero_shot_en_to_sl_confusion_matrix.png


## Few-shot adaptation (mBERT)

Fine-tune the mBERT model (trained on English) on a small Slovenian sample to adapt it.

**Validation holdout suggestion:** when using very small few-shot sizes, keep a small validation holdout (e.g. 10-20% of the few-shot examples or 50–100 examples) to monitor overfitting. The code below will automatically use the existing Slovenian validation split if available, otherwise it will create a holdout from the sampled few-shot set.

In [ ]:
# Few-shot evaluation and comparison using the existing build_trainer helper
import importlib
import util
importlib.reload(util)
from util import load_datasets_from_hf, tokenize_dataset, load_model_and_tokenizer, load_trained_model_and_tokenizer, build_trainer, predict_labels
import os
from datetime import datetime
import csv
import graphs

few_shot_ks = [0, 10, 50, 100, 200]
mb_model_name = "bert-base-multilingual-cased"
trained_root = "./trained_models"
results_dir = "./results"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(trained_root, exist_ok=True)
label2id = {"ham": 0, "spam": 1}

# Ensure English and Slovenian splits are available
try:
    train_dataset
    val_dataset
    test_dataset
except NameError:
    label2id = {"ham": 0, "spam": 1}
    train_dataset, val_dataset, test_dataset, id2label = load_datasets_from_hf("daviiiidcpp1/sms-spam-combined", split="train", label2id=label2id)

try:
    sl_train_ds
    sl_val_ds
    sl_test_ds
    sl_id2label
except NameError:
    sl_train_ds, sl_val_ds, sl_test_ds, sl_id2label = load_datasets_from_hf("daviiiidcpp1/sms-spam-slovene", split="train", label2id=label2id, train_size=0.8, val_size=0.1, seed=42)

# Prefer the explicitly named saved mBERT checkpoint; otherwise train a fresh English base.
base_model_name = "bert-base-multilingual-cased2026-05-07_22-06-34"
base_dir = os.path.join(trained_root, base_model_name)

if os.path.isdir(base_dir):
    model, tokenizer = load_trained_model_and_tokenizer(base_dir, num_labels=2)
    print("Using saved mBERT base checkpoint:", base_dir)
else:
    model, tokenizer = load_model_and_tokenizer(mb_model_name, num_labels=2)
    train_tok = tokenize_dataset(train_dataset, tokenizer)
    val_tok = tokenize_dataset(val_dataset, tokenizer)
    base_trainer = build_trainer(
        model,
        train_tok,
        val_tok,
        learning_rate=2e-5,
        num_epochs=1,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        output_dir=os.path.join(results_dir, mb_model_name.split("/")[-1] + "_base"),
        logging_steps=50,
        save_total_limit=1,
    )
    base_trainer.train()
    base_dir = os.path.join(trained_root, mb_model_name.split("/")[-1] + "_base_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))
    base_trainer.save_model(base_dir)
    tokenizer.save_pretrained(base_dir)
    print("Trained and saved English base mBERT to:", base_dir)

# Tokenize Slovenian test once for all runs
sl_test_ds_tok_full = tokenize_dataset(sl_test_ds, tokenizer)
results_csv = os.path.join(results_dir, mb_model_name.split("/")[-1] + "_fewshot_results.csv")
labels_in_order = [sl_id2label[i] for i in range(len(sl_id2label))]

with open(results_csv, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["few_shot_k", "accuracy", "precision", "recall", "f1", "train_size", "val_size", "checkpoint_dir"])

for k in few_shot_ks:
    print(f"--- few_shot_k={k} ---")
    if k == 0:
        base_model, base_tokenizer = load_trained_model_and_tokenizer(base_dir, num_labels=2)
        base_eval_trainer = build_trainer(
            base_model,
            train_tok if 'train_tok' in globals() else tokenize_dataset(train_dataset, base_tokenizer),
            val_tok if 'val_tok' in globals() else tokenize_dataset(val_dataset, base_tokenizer),
            output_dir=os.path.join(results_dir, mb_model_name.split("/")[-1] + "_zero_shot_eval"),
        )
        base_eval_trainer.eval_dataset = sl_test_ds_tok_full
        metrics = base_eval_trainer.evaluate()
        preds, labels = predict_labels(base_eval_trainer, sl_test_ds_tok_full)
        save_name = mb_model_name.split("/")[-1] + "_zero_shot_eval_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        graph_dir = os.path.join("./graphs", save_name)
        os.makedirs(graph_dir, exist_ok=True)
        graphs.plot_confusion_matrix(y_true=labels, y_pred=preds, labels=labels_in_order, save_path=os.path.join(graph_dir, "confusion_matrix.png"))
        with open(results_csv, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow([k, metrics["eval_accuracy"], metrics["eval_precision"], metrics["eval_recall"], metrics["eval_f1"], 0, len(sl_val_ds), base_dir])
        continue

    k_use = min(k, len(sl_train_ds))
    few_shot_train = sl_train_ds.shuffle(seed=42).select(range(k_use))

    if len(sl_val_ds) >= 20:
        few_shot_val = sl_val_ds.shuffle(seed=43).select(range(min(100, len(sl_val_ds))))
    else:
        holdout = max(1, int(0.1 * k_use))
        holdout = min(holdout, max(0, k_use - 1))
        if holdout > 0:
            few_shot_val = few_shot_train.select(range(holdout))
            few_shot_train = few_shot_train.select(range(holdout, k_use))
        else:
            few_shot_val = few_shot_train

    few_shot_train_tok = tokenize_dataset(few_shot_train, tokenizer)
    few_shot_val_tok = tokenize_dataset(few_shot_val, tokenizer)

    model_k, tokenizer_k = load_trained_model_and_tokenizer(base_dir, num_labels=2)
    trainer_k = build_trainer(
        model_k,
        few_shot_train_tok,
        few_shot_val_tok,
        learning_rate=2e-5,
        num_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        output_dir=os.path.join(results_dir, f"{mb_model_name.split('/' )[-1]}_fewshot_k{k}"),
        logging_steps=50,
        save_total_limit=2,
    )
    trainer_k.train()
    trainer_k.eval_dataset = sl_test_ds_tok_full
    metrics_k = trainer_k.evaluate()
    preds_k, labels_k = predict_labels(trainer_k, sl_test_ds_tok_full)

    save_name_k = mb_model_name.split("/")[-1] + f"_fewshot_k{k}_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    save_dir_k = os.path.join(trained_root, save_name_k)
    trainer_k.save_model(save_dir_k)
    tokenizer.save_pretrained(save_dir_k)

    graph_dir_k = os.path.join("./graphs", save_name_k)
    os.makedirs(graph_dir_k, exist_ok=True)
    graphs.plot_confusion_matrix(y_true=labels_k, y_pred=preds_k, labels=labels_in_order, save_path=os.path.join(graph_dir_k, "confusion_matrix.png"))

    with open(results_csv, "a", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow([k, metrics_k["eval_accuracy"], metrics_k["eval_precision"], metrics_k["eval_recall"], metrics_k["eval_f1"], len(few_shot_train_tok), len(few_shot_val_tok), save_dir_k])

    print("Saved checkpoint to:", save_dir_k)
    print("Saved confusion matrix to:", os.path.join(graph_dir_k, "confusion_matrix.png"))

print("All few-shot runs completed. Results saved to:", results_csv)

Loaded datasets (train=44323, val=5540, test=5541)
Loaded datasets (train=44323, val=5540, test=5541)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Using saved mBERT base checkpoint: ./trained_models/bert-base-multilingual-cased2026-05-07_22-06-34
--- few_shot_k=0 ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/44323 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.000046,0.356801,3,0.963003,0.938149,0.964095,0.950945


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint to: ./trained_models/bert-base-multilingual-cased_fewshot_k200_2026-05-08_14-15-49
Saved confusion matrix to: ./graphs/bert-base-multilingual-cased_fewshot_k200_2026-05-08_14-15-49/confusion_matrix.png
All few-shot runs completed. Results saved to: ./results/bert-base-multilingual-cased_fewshot_results.csv


In [3]:
import importlib
import graphs

importlib.reload(graphs)

metrics = ["accuracy", "precision", "recall", "f1"]
few_shot_results_csv = os.path.join("./results", "bert-base-multilingual-cased_fewshot_results.csv")

for metric in metrics:
    plot_path = graphs.plot_fewshot_comparison(few_shot_results_csv, save_path="./graphs", metric=metric)
    print(f"Saved few-shot comparison plot for {metric} to:", plot_path)

Saved few-shot comparison plot for accuracy to: ./graphs/fewshot_comparison_accuracy.png
Saved few-shot comparison plot for precision to: ./graphs/fewshot_comparison_precision.png
Saved few-shot comparison plot for recall to: ./graphs/fewshot_comparison_recall.png
Saved few-shot comparison plot for f1 to: ./graphs/fewshot_comparison_f1.png
